# DBAS 5015 — Week 9 Lab: Final Project Notebook Template

**Course:** Introduction to Data Science  
**Week:** 9 — Full Workflow Integration  
**Estimated time:** Self-paced (this is your final project)

---

## How to Use This Template

This notebook is a scaffold for your final project. Every section has:

- A **guidance cell** (like this one) explaining what to do and why
- One or more **code cells** with starter code you should replace with your own analysis

**Instructions:**
1. Read each guidance cell before writing any code in that section
2. Replace placeholder code with your own — delete anything that doesn't apply to your dataset
3. Add markdown cells to explain your reasoning whenever you make a non-obvious decision
4. When the notebook is complete, run all cells from top to bottom (Kernel → Restart & Run All) to confirm it executes without errors

> 🔁 **Default dataset:** This template ships with a loan application dataset so you can see how the structure works. Replace it with your own data in Part 1 — every downstream section will adapt naturally.

---
## Part 0: Setup and Imports

Import all libraries you will need for the project here. Having all imports at the top makes your notebook easier to debug and reproduce.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
# Regression models (keep what you need, delete the rest)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
# Classification models (keep what you need, delete the rest)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
# Clustering (keep what you need, delete the rest)
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             confusion_matrix, classification_report, silhouette_score)
import warnings
warnings.filterwarnings('ignore')
print('All imports successful.')

---
## Part 1: Dataset and Problem Definition

### 1.1 Load Your Dataset

Replace the default loan dataset below with your own data. If loading from a CSV file, use `pd.read_csv('your_file.csv')`. After loading, confirm the shape and peek at the first few rows.

In [ ]:
# DEFAULT DATASET: Loan Applications (synthetic)
# REPLACE this block with your own pd.read_csv() or data loading code.
import numpy as np
np.random.seed(99)
n = 500
df = pd.DataFrame({
    'age':               np.random.randint(22, 65, n),
    'annual_income':      np.random.normal(55000, 18000, n).clip(15000, 150000).round(-2),
    'loan_amount':        np.random.normal(12000, 5000, n).clip(2000, 40000).round(-2),
    'loan_term_months':   np.random.choice([12, 24, 36, 48, 60], n),
    'credit_score':       np.random.randint(500, 800, n),
    'employment_type':    np.random.choice(['Full-time', 'Part-time', 'Self-employed', 'Unemployed'], n,
                                           p=[0.60, 0.15, 0.18, 0.07]),
    'num_existing_loans': np.random.randint(0, 5, n),
    'has_collateral':     np.random.choice([0, 1], n, p=[0.55, 0.45]),
})
default_prob = (1 - (df['credit_score'] - 500) / 300) * 0.5 + (df['num_existing_loans'] / 4) * 0.3
default_prob = default_prob.clip(0.05, 0.90)
df['defaulted'] = (np.random.rand(n) < default_prob).astype(int)
missing_idx = np.random.choice(df.index, size=25, replace=False)
df.loc[missing_idx[:15], 'annual_income'] = np.nan
df.loc[missing_idx[15:], 'credit_score'] = np.nan
print(f'Dataset shape: {df.shape}')
df.head()

### 1.2 Problem Statement

**Write your problem statement here before writing any further code.**

A good problem statement answers three questions:
1. What domain is this data from?
2. What are you trying to predict or discover?
3. Why does this matter — what decision will the result support?

*Example (for the default loan dataset):*
> "This project uses loan application records to predict whether an applicant will default on their loan. The goal is to help a lending institution flag high-risk applications for additional underwriting review, reducing financial losses while maintaining approval rates for low-risk applicants."

---

**Your problem statement:**

*[Replace this text with your own problem statement.]*

---
## Part 2: Exploratory Data Analysis (EDA)

EDA is not optional — it is where you learn whether your data can actually answer your question.

### 2.1 Data Overview

In [ ]:
print('Shape:', df.shape)
print()
print('Column types:')
print(df.dtypes)
print()
print('Missing values:')
print(df.isnull().sum())

In [ ]:
df.describe().round(2)

### 2.2 Target Variable Distribution

For supervised learning: visualize the distribution of your target variable.
For classification: check class balance. For regression: check for outliers. For clustering: skip to 2.3.

In [ ]:
target_col = 'defaulted'  # REPLACE with your target column name
counts = df[target_col].value_counts().reset_index()
counts.columns = [target_col, 'count']
counts['proportion'] = (counts['count'] / len(df) * 100).round(1)
fig = px.bar(counts, x=target_col, y='count',
             text=counts['proportion'].astype(str) + '%',
             title=f'Target Variable: {target_col} Distribution',
             labels={target_col: 'Class', 'count': 'Number of Records'})
fig.update_traces(textposition='outside')
fig.show()

### 2.3 Feature Distributions

Visualize key numeric and categorical features. Look for skew, outliers, and unusual values.

In [ ]:
numeric_cols_check = ['annual_income', 'credit_score', 'loan_amount', 'age']  # adjust
for col in numeric_cols_check:
    fig = px.histogram(df, x=col, nbins=30, title=f'Distribution: {col}')
    fig.show()

In [ ]:
cat_cols_check = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols_check:
    counts = df[col].value_counts().reset_index()
    counts.columns = [col, 'count']
    fig = px.bar(counts, x=col, y='count', title=f'Category Counts: {col}')
    fig.show()

### 2.4 Feature–Target Relationships

In [ ]:
target_col = 'defaulted'
for col in ['annual_income', 'credit_score', 'loan_amount']:
    fig = px.box(df, x=target_col, y=col,
                 title=f'{col} by {target_col}',
                 labels={str(target_col): 'Defaulted (0=No, 1=Yes)'})
    fig.show()

In [ ]:
target_col = 'defaulted'
cat_col = 'employment_type'  # replace with your categorical feature
rate = df.groupby(cat_col)[target_col].mean().reset_index()
rate.columns = [cat_col, 'default_rate']
rate['default_rate_pct'] = (rate['default_rate'] * 100).round(1)
fig = px.bar(rate, x=cat_col, y='default_rate_pct',
             title=f'Default Rate by {cat_col}',
             labels={'default_rate_pct': 'Default Rate (%)'})
fig.add_hline(y=df[target_col].mean() * 100, line_dash='dash',
              annotation_text='Overall Average')
fig.show()

### 2.5 EDA Summary

**After completing your EDA, write 3–5 observations here.**

Consider: What did you learn about the data? Which features seem most predictive? Were there data quality issues? What preprocessing decisions will these findings drive?

*Example:*
> 1. The dataset has a 27% default rate — moderately imbalanced but manageable.
> 2. Credit score and annual income are the strongest separators between defaulters and non-defaulters.
> 3. 15 records are missing annual_income and 10 are missing credit_score — both will be imputed with the median.
> 4. Employment type shows meaningful variation — self-employed and unemployed applicants default roughly twice as often as full-time employees.

---

**Your EDA observations:**

1.  
2.  
3.  
4.  
5.  

---
## Part 3: Preprocessing and Pipeline

### 3.1 Define Feature Groups

Assign each feature to the appropriate preprocessing strategy. Every feature you plan to use must appear in exactly one list.

In [ ]:
# REPLACE with your actual column names
numeric_cols      = ['age', 'annual_income', 'loan_amount', 'credit_score']
categorical_cols  = ['employment_type']
passthrough_cols  = ['has_collateral', 'num_existing_loans', 'loan_term_months']
target_col        = 'defaulted'
all_feature_cols  = numeric_cols + categorical_cols + passthrough_cols
print('Features:', all_feature_cols)
print('Target:  ', target_col)

### 3.2 Train/Test Split

For supervised learning only. Always split before fitting any transformer.
For clustering, skip this cell.

In [ ]:
X = df[all_feature_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training set: {X_train.shape[0]} rows')
print(f'Test set:     {X_test.shape[0]} rows')
print(f'Train target rate: {y_train.mean():.3f}')
print(f'Test target rate:  {y_test.mean():.3f}')

### 3.3 Build the ColumnTransformer

In [ ]:
from sklearn.pipeline import Pipeline as SKPipeline
numeric_pipeline = SKPipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
preprocessor = ColumnTransformer(transformers=[
    ('numeric',     numeric_pipeline,                                 numeric_cols),
    ('categorical', OneHotEncoder(handle_unknown='ignore',
                                  sparse_output=False),               categorical_cols),
    ('passthrough', 'passthrough',                                    passthrough_cols)
])
X_check = preprocessor.fit_transform(X_train)
print(f'Preprocessed shape: {X_check.shape}')
print('Preprocessing looks good!')

---
## Part 4: Modelling

### 4.1 Baseline Model

Start with a simple model. A baseline gives you a point of comparison.

In [ ]:
pipe_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=500, random_state=42))
])
cv_scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=5, scoring='f1')
print(f'Logistic Regression  --  CV F1: {cv_scores_lr.mean():.3f} +/- {cv_scores_lr.std():.3f}')

### 4.2 Second Model

Train a more complex model and compare using the same cross-validation setup.

In [ ]:
pipe_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])
cv_scores_rf = cross_val_score(pipe_rf, X_train, y_train, cv=5, scoring='f1')
print(f'Random Forest  --  CV F1: {cv_scores_rf.mean():.3f} +/- {cv_scores_rf.std():.3f}')
comparison = pd.DataFrame({
    'Model':       ['Logistic Regression', 'Random Forest'],
    'CV F1 Mean':  [cv_scores_lr.mean(), cv_scores_rf.mean()],
    'CV F1 Std':   [cv_scores_lr.std(),  cv_scores_rf.std()]
}).round(3)
print(comparison.to_string(index=False))

### 4.3 Hyperparameter Tuning

Tune your best model using GridSearchCV. The search runs only on training data.

In [ ]:
param_grid = {
    'model__n_estimators':    [100, 200],
    'model__max_depth':       [None, 5, 10],
    'model__min_samples_leaf': [1, 5]
}
grid_search = GridSearchCV(pipe_rf, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)
print(f'Best CV F1:  {grid_search.best_score_:.3f}')
print(f'Best params: {grid_search.best_params_}')

---
## Part 5: Evaluation on the Test Set

**This section runs exactly once.** You have finished tuning — now evaluate your final model on held-out data.

### 5.1 Final Model Evaluation

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print('=== Final Test Set Evaluation ===')
print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))
cm = confusion_matrix(y_test, y_pred)
fig = px.imshow(cm,
                labels=dict(x='Predicted', y='Actual', color='Count'),
                x=['No Default', 'Default'],
                y=['No Default', 'Default'],
                text_auto=True,
                title='Confusion Matrix — Final Model (Test Set)',
                color_continuous_scale='Blues')
fig.show()

### 5.2 Feature Importances

In [ ]:
ohe_features = (best_model.named_steps['preprocessor']
                .named_transformers_['categorical']
                .get_feature_names_out(categorical_cols).tolist())
all_feature_names = numeric_cols + ohe_features + passthrough_cols
importances = best_model.named_steps['model'].feature_importances_
imp_df = pd.DataFrame({'feature': all_feature_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=True).tail(15)
fig = px.bar(imp_df, x='importance', y='feature', orientation='h',
             title='Top Feature Importances — Final Model')
fig.show()

### 5.3 Interpretation

**Write your interpretation here.**

For each of the top 3–5 features, write one sentence explaining what the importance means in plain language in the context of your specific problem.

*Example:*
> - **Credit score** (importance 0.31): The most influential predictor — applicants with lower credit scores default at substantially higher rates.
> - **Annual income** (importance 0.24): Higher income reduces default risk, likely because these applicants have more financial buffer.
> - **Num existing loans** (importance 0.18): Applicants carrying more debt are more likely to default, suggesting overleveraging is a meaningful risk signal.

---

**Your interpretation:**

-  
-  
-  

---
## Part 6: Executive Summary (Draft)

Write a plain-language summary of your project findings here. This will become the basis for your `executive_summary.pdf` or `.docx` submission.

**Requirements:**
- No code references
- No technical jargon without plain-English explanation
- A concrete, actionable recommendation
- One paragraph on limitations or caveats

---

### [Your Project Title Here]

**Prepared by:** [Your Name]  
**Date:** [Date]

**The Question**

*[One paragraph: what problem did you set out to solve, and why does it matter?]*

**What We Found**

*[2–3 paragraphs: key findings in plain language. Paste 2–3 charts below this cell.]*

**Our Recommendation**

*[1 paragraph: what specific action should a decision-maker take based on these findings?]*

**Limitations and Next Steps**

*[1 paragraph: limitations of this analysis and what would make it stronger.]*

In [ ]:
# Re-run your 2-3 best charts here for the executive summary section
fig = px.bar(imp_df, x='importance', y='feature', orientation='h',
             title='Key Drivers — Final Model')
fig.show()

---
## Challenge: Risk Tiers or Segment Profiles

Choose the challenge that matches your project type.

### Challenge A — Classification: Risk Tiers

Use `predict_proba` to assign records to risk tiers, then validate that tiers have monotonically increasing actual rates.

In [ ]:
proba    = best_model.predict_proba(X_test)[:, 1]
tier_df  = X_test.copy()
tier_df['prob_positive'] = proba
tier_df['actual']        = y_test.values
tier_df['risk_tier']     = pd.cut(
    tier_df['prob_positive'],
    bins=[0, 0.25, 0.50, 0.75, 1.01],
    labels=['Low', 'Medium', 'High', 'Critical']
)
validation = tier_df.groupby('risk_tier', observed=True).agg(
    count=('actual', 'count'),
    actual_rate=('actual', 'mean')
).round(3)
print(validation)
print('Monotonically increasing?',
      all(validation['actual_rate'].diff().dropna() >= 0))

In [ ]:
fig = px.box(tier_df, x='risk_tier', y='prob_positive',
             title='Predicted Probability Distribution by Risk Tier',
             category_orders={'risk_tier': ['Low','Medium','High','Critical']})
fig.show()

### Challenge B — Clustering: Segment Profiles

Create a detailed profile of each segment. Give each segment a descriptive business name.

In [ ]:
# Replace with your clustering code if doing unsupervised learning
X_cluster = df[numeric_cols].copy()
scaler_c  = StandardScaler()
X_scaled  = scaler_c.fit_transform(X_cluster.fillna(X_cluster.median()))
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df['segment'] = km.fit_predict(X_scaled)
profile = df.groupby('segment')[numeric_cols].mean().round(1)
print('Segment Profiles:')
print(profile)
segment_names = {0: 'Segment A', 1: 'Segment B', 2: 'Segment C'}
df['segment_name'] = df['segment'].map(segment_names)
print()
print('Segment sizes:')
print(df['segment_name'].value_counts())

---
## Submission Checklist

Before submitting, confirm all items below:

- [ ] Notebook runs top-to-bottom without errors (Kernel → Restart & Run All)
- [ ] Problem statement written in plain English (Section 1.2)
- [ ] EDA includes at least 4 Plotly visualizations
- [ ] Missing values documented and handled
- [ ] ColumnTransformer fit only on training data (or full data for clustering)
- [ ] Pipeline used throughout
- [ ] At least two models trained and compared with cross-validation
- [ ] Hyperparameter tuning performed with GridSearchCV or RandomizedSearchCV
- [ ] Final model evaluated on test set exactly once
- [ ] Metrics interpreted in plain language (Section 5.3)
- [ ] Executive summary draft written in Section 6
- [ ] Challenge section completed (A or B)
- [ ] All variable names are meaningful
- [ ] Executive summary submitted as a separate PDF or DOCX
- [ ] Dataset or source citation included in submission folder